In [2]:
import pandas as pd


In [8]:
df = pd.read_csv('../../log/evaluation.csv')

In [5]:
df

,audio,reference,prediction,match,WER,CER,latency
0,2128_audio01.wav,"Aku mau book flight ke Jeddah minggu depan, bi...",Aku mau book flag ke Jeddah minggu depan. Bisa...,False,0.273,0.078,32.20
1,2128_audio02.wav,Aku butuh travel umrah simple tapi include Mad...,"Aku butuh travel umroh simple, tapi ikut Madin...",False,0.444,0.143,31.21
2,2128_audio03.wav,Can you help aku arrange transport dari Jeddah...,Can you help aku arrange transfer dari Jeddah ...,False,0.182,0.061,31.01
3,2128_audio04.wav,Explain step by step cara apply visa Saudi den...,Explain step by step cara apply visa Saudi den...,False,0.100,0.018,30.29
4,2128_audio05.wav,"Ya akhi, uridu book flight ila Jeddah al-usbu ...","Ya al-Qi, Uridu Bukflek ilah Jeddah al-Usbu' a...",False,0.778,0.250,37.91
...,...,...,...,...,...,...,...
556,2379_audio07.wav,"Book flight ke Jeddah lalu lanjut ke Madinah, ...",Book flight ke Jeddah lalu lanjut ke Madinah. ...,False,0.273,0.044,39.60
557,2379_audio12.wav,Bagaimana proses visa Saudi untuk umrah dari I...,Bagaimana proses visa Saudi untuk umroh dari I...,False,0.222,0.032,37.76
558,2379_audio18.wav,"I feel overwhelmed dengan persiapan umrah, ada...",I feel overwhelmed dengan persiapan umroh. Ada...,False,0.222,0.048,38.70
559,2479_audio04.wav,Explain step by step cara apply visa Saudi den...,Explain step by step cara apply visa Saudi den...,False,0.100,0.018,37.95


In [9]:

# threshold
WER_THRESHOLD = 0.9
CER_THRESHOLD = 0.9

# cari sample error tinggi
high_error = df[
    (df["WER"] >= WER_THRESHOLD) |
    (df["CER"] >= CER_THRESHOLD)
]

# tampilkan hasil
print("=" * 60)
print("HIGH ERROR SAMPLES")
print("=" * 60)

for _, row in high_error.iterrows():

    print(f"\nAudio      : {row['audio']}")
    print(f"WER        : {row['WER']}")
    print(f"CER        : {row['CER']}")
    print(f"Latency    : {row['latency']} sec")

    # optional jika ada kolom reference/prediction
    if "reference" in df.columns:
        print(f"Reference  : {row['reference']}")

    if "prediction" in df.columns:
        print(f"Prediction : {row['prediction']}")

    # analisis sederhana
    if row["WER"] > 1.0:
        print("Analysis   : Severe transcription failure")

    elif row["CER"] > 0.7:
        print("Analysis   : Heavy character-level error")

    else:
        print("Analysis   : Moderate code-switching error")

print("\n")
print(f"Total high-error samples: {len(high_error)}")


HIGH ERROR SAMPLES

Audio      : 2313_audio05.wav
WER        : 1.0
CER        : 0.741
Latency    : 32.4 sec
Reference  : Ya akhi, uridu book flight ila Jeddah al-usbu al-qadim. Hal bisa bantu ajida afdhal schedule wa rihlatan mubashirah?
Prediction : Sampai jumpa di video selanjutnya, dan sampai jumpa di video selanjutnya.
Analysis   : Heavy character-level error

Audio      : 2310_Audio20.wav
WER        : 1.0
CER        : 0.776
Latency    : 31.84 sec
Reference  : Translate ke English: aku mau pergi ke Makkah minggu depan
Prediction : I want to go to Mecca next week.
Analysis   : Heavy character-level error

Audio      : 2318_audio05.wav
WER        : 1.0
CER        : 0.784
Latency    : 33.15 sec
Reference  : Ya akhi, uridu book flight ila Jeddah al-usbu al-qadim. Hal bisa bantu ajida afdhal schedule wa rihlatan mubashirah?
Prediction : Terima kasih telah menonton. Terima kasih telah menonton. Terima kasih telah menonton. Terima kasih.
Analysis   : Heavy character-level error

Audio    

In [3]:
import json
import re

# function hapus <think>...</think>
def clean_think(text):

    text = re.sub(
        r"<think>.*?</think>",
        "",
        text,
        flags=re.DOTALL
    )

    return text.strip()

# baca json
with open(
    "../../log/llm_output.json",
    "r",
    encoding="utf-8"
) as f:

    data = json.load(f)

# bersihkan semua response
for item in data:

    item["preserve_response"] = clean_think(
        item["preserve_response"]
    )

    item["normalized_response"] = clean_think(
        item["normalized_response"]
    )

# lihat hasil pertama
print(
    json.dumps(
        data[0],
        indent=4,
        ensure_ascii=False
    )
)

{
    "audio": "2128_audio01.wav",
    "input_text": "Aku mau book flag ke Jeddah minggu depan. Bisa bantu schedule?",
    "preserve_response": "Aku bisa bantu, tapi jelasin dulu maksud \"book flag\"-nya. Maksudnya booking flight ke Jeddah? Kalo iya, tgl berapa minggu depan? Dan butuh bantuan buat visa, hotel, atau apa aja nanti? 😊",
    "normalized_response": "Tentu, berikut langkah-langkah untuk membantu menyiapkan jadwal perjalanan Anda ke Jeddah minggu depan:\n\n1. **Konfirmasi Tanggal dan Waktu**  \n   - \"Minggu depan\" dalam bahasa Indonesia umumnya merujuk pada **Senin hingga Minggu**. Mohon spesifikasikan hari dan tanggal pasti (contoh: 3-7 Mei 2024) untuk mempermudah pencarian tiket.\n\n2. **Kota Asal**  \n   - Beri tahu kota/pelabuhan keberangkatan Anda (contoh: Jakarta, Surabaya, Makassar). Jadwal dan harga tiket bervariasi tergantu kota asal.\n\n3. **Pilihan Maskapai & Klasifikasi**  \n   - Apakah Anda menginginkan penerbangan **langsung** atau **transit**?  \n   - Kebutuh

In [5]:
with open(
    "../../log/llm_output_clean.json",
    "w",
    encoding="utf-8"
) as f:

    json.dump(
        data,
        f,
        indent=4,
        ensure_ascii=False
    )